In [ ]:
def solve_dispersive_pde_equation(
    L=100, c=1, k=12,
    boundary="absorbing",
    f=lambda x, t: 0,
    omega = .99,
    u_0=lambda x: np.exp(-x**2),
    v_0=lambda x: 0,
    g=lambda u: 0,
    T = 2*np.pi / (np.sqrt(np.pi**2+1))
):
    a = -L
    b = L
    T = 2*np.pi / (np.sqrt(np.pi**2+1))
    M = 2**k

    bd = boundary

    h = (b - a) / M

    cfl = .9

    N = int(c * T / (cfl*h))
    tau = T / N

    xs = np.linspace(a, b, M + 1)
    ts = np.linspace(0, T, N + 1)

    U = np.zeros((M + 1, N + 1))

    U[:, 0] = u_0(xs)

    # boundary conditions at t_0
    U[0, 0] = u_0(xs[0])
    U[M, 0] = u_0(xs[M])  # U[0, 0]

    # At t_1
    U[1:M, 1] = (
        U[1:M, 0]
        + tau * v_0(xs[1:M])
        + tau**2 * c**2 / (2 * h**2) * (U[2:M+1, 0] - 2 * U[1:M, 0] + U[0:M-1, 0])
        + tau**2 / 2 * f(xs[1:M], ts[0])
        - tau**2 / 2 * g(U[1:M, 0])
    )

    # absorbing boundary conditions at t_1
    if boundary == "absorbing":
        U[0, 1] = U[1, 0]
        U[M, 1] = U[M-1, 0]

    # periodic boundary conditions at t_1
    if boundary == "periodic":
        U[0, 1] = (
            U[0, 0] + tau * v_0(xs[0])
            + tau**2 * c**2 / (2 * h**2) * (U[1, 0] - 2 * U[0, 0] + U[M - 1, 0])
            + tau**2 / 2 * f(xs[0], ts[0]) - tau**2 / 2 * g(U[0, 0])
        )
        U[M, 1] = U[0, 1]

    coeff = tau**2 * c**2 / h**2

    # explicit timestepping
    for n in range(1, N):
        U[1:M, n + 1] = (
            2 * U[1:M, n]
            - U[1:M, n - 1]
            + coeff * (U[2:M+1, n] - 2 * U[1:M, n] + U[0:M-1, n])
            + tau**2 * f(xs[1:M], ts[n])
            - tau**2 * g(U[1:M, n])
        )

        # absorbing boundary conditions at t_1
        if boundary == "absorbing":
            U[0, n+1] = U[1, n]
            U[M, n+1] = U[M-1, n]

        if boundary == "periodic":
            U[0, n + 1] = (
                2 * U[0, n]
                - U[0, n - 1]
                + coeff * (U[1, n] - 2 * U[0, n] + U[M - 1, n])
                + tau**2 * f(xs[0], ts[n]) - tau**2 * g(U[0, n])
            )
            U[M, n + 1] = U[0, n + 1]

    return U, xs, ts
def animate_over_circle(t_values,s_values,u_values):
    fig = plt.figure()
    ax = fig.add_subplot(projection='3d')
    ax.set_title('t = {:.2f}'.format(t_values[0]))
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_zlabel('u')
    z_min = min(min(u_value) for u_value in u_values)
    z_min -= 0.1 if abs(z_min) < 0.1 else 0.1 * abs(z_min)
    z_max = max(max(u_value) for u_value in u_values)
    z_max += 0.1 if abs(z_max) < 0.1 else 0.1 * abs(z_max)
    ax.set_zlim(z_min,z_max)
    L = -s_values[0]
    r = L
    
    
    xs=[L*np.cos(s/r)for s in s_values]
    ys=[L*np.sin(s/r)for s in s_values]
    
    line = ax.plot(xs, ys, u_values[0])[0]
    def animate(frame):
        ax.set_title('t = {:.2f}'.format(t_values[frame]))
        line.set_data_3d(xs, ys, u_values[frame])
        return line
    anim = animation.FuncAnimation(fig,animate,frames=len(u_values),interval=30)
    plt.close()
    video = HTML(anim.to_html5_video())
    return video

L=100
a = -L
b = L
c = 1
T = 100
k= 12
cfl = .9
M = 2**k
h = (b - a) / (M)
N = int(c * T / (cfl*h))
tau = T / (N)
omega= .99
u_0=lambda x: np.exp(-x**2)
v_0=lambda x: 0
g=lambda u: 0


U, x ,y = solve_dispersive_pde_equation(L,c,k,"absorbing",f,omega,u_0,v_0,g,T)
u_values = [U[:,n]for n in range(U.shape[1])]
animate_over_circle(y,x,u_values)